# Mining comparison, Phase 4: mini experiment per method

Does every mining implementation produce usable features on the real data?
Each method mines one window, the precision gate (g = 2) and the budget K are
applied, and the features are encoded on the next window. No model is trained:
this is a working-implementation check, not the comparison itself.

- **CSCAS:** W0 → W1 of the paper's walk-forward (baseline split, γ = 0.05; same windows as Checks 1–3), oracle fields excluded.
- **AIT-ADS:** fox, cscas_grouping, first 70% of groups → the rest. Look-back sequences: w = 10 min, 1-minute buckets.
- **Temporal methods on CSCAS are n/a.** CSCAS has no raw per-alert stream to build sequences from.

**Pass (per method × dataset):** for each class, at least one budgeted feature that fires on the test window and is not a single-signature restatement (it fires on groups of more than one signature in the mining window).

Logic: `thesis.system_eval.mining_mini_experiment`. Run: `scripts/system_eval/run_mining_mini_experiment.py`.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from thesis.paths import ROOT

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)

## Load

In [ ]:
RUN_DIR = None  # None -> latest run, else a path

def _latest_run() -> Path:
    base = ROOT / "artifacts" / "experiments" / "mining_mini"
    runs = sorted(d for d in base.glob("*") if (d / "summary.csv").exists()) if base.exists() else []
    if not runs:
        raise FileNotFoundError("no mining_mini run found")
    return runs[-1]

run_dir = Path(RUN_DIR) if RUN_DIR else _latest_run()
summary = pd.read_csv(run_dir / "summary.csv")
features = pd.read_csv(run_dir / "features.csv")
print(run_dir, "|", len(summary), "cells |", len(features), "budgeted features")

## Verdict per method × dataset

In [ ]:
cols = ["dataset", "method", "status", "n_train", "n_train_attack", "mine_seconds",
        "n_raw", "n_gated", "n_budget", "n_budget_attack", "n_budget_benign"]
summary[[c for c in cols if c in summary]]

## Are the features usable?

- `dead_on_train` should be 0: a feature that never fires on the window it was mined on means mining and encoding disagree.
- `live_share_test`: share of features that fire at least once on the test window.
- `single_signature_share`: share of features that only restate one signature.
- Coverage: share of test attacks (benign groups) with at least one attack-leaning (benign-leaning) feature firing.

In [ ]:
cols = ["dataset", "method", "dead_on_train", "live_share_test", "single_signature_share",
        "test_attack_coverage", "test_benign_coverage",
        "median_test_attack_rate_attack_feats", "test_base_rate",
        "mean_conditions", "max_conditions"]
summary[[c for c in cols if c in summary]].dropna(subset=["live_share_test"])

## Top features per method, dataset and class (by mining-window support)

In [ ]:
def top(df: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    return (df.sort_values("support", ascending=False).head(n)
              [["condition", "support", "train_attack_rate", "test_fires",
                "test_attack_rate", "single_signature"]])

for (dataset, method, label), grp in features.groupby(["dataset", "method", "source_label"]):
    print(f"\n=== {dataset} | {method} | {label}-leaning ({len(grp)} features)")
    display(top(grp))

## Restatements and dead features, listed

In [ ]:
features[features["single_signature"] | ~features["live_test"]][
    ["dataset", "method", "source_label", "condition", "train_fires", "test_fires", "single_signature"]
]